# Chen 2024 external validation — canonical notebook

Self-contained. Installs ANARCI (pip), downloads the Chen 2024 data and the checksum-verified Model 1 coefficients (pinned to the release tag), annotates 80,000 antibodies with ANARCI, then reproduces:

1. the **all-79,999 frozen** zero-shot transfer score (reported parenthetically in the manuscript); and
2. the **common held-out test set** (n=16,000) on which every Figure 5A,B bar is evaluated (frozen model, refit models, and charge baselines all on the same stratified 20% test split), with bootstrap CIs and top-k precision reported directly.

**Run:** Runtime -> Run all (CPU). ~15 min. Outputs three JSON files: `chen_anarci_results.json` (all-data frozen and baseline results), `chen_fig5_common_test.json` (common held-out test results), and `chen_fig5_merged.json` (the combined file read by make_figure5.py)`. This is the single canonical reproduction notebook for Section 3.8 and Figure 5.

In [ ]:
# Cell 1 — pip-based ANARCI install (no conda, no restart)
import sys, subprocess

# 1) HMMER: the one non-python dependency ANARCI needs
!apt-get -qq install -y hmmer > /dev/null 2>&1
print("hmmer:", subprocess.run(["which","hmmscan"],capture_output=True,text=True).stdout.strip() or "NOT FOUND")

# 2) ANARCI + abnumber from pip. The modern PyPI wheel (jnooree fork) is pure-python.
!pip install -q anarci==1.3 abnumber==0.3.4

# 3) verify import; if the wheel didn't provide the germline HMMs, fall back to GitHub source
try:
    from abnumber import Chain
    print("abnumber import OK (pip)")
except Exception as e:
    print("pip import failed, installing ANARCI from GitHub source:", e)
    !pip install -q biopython
    !git clone -q https://github.com/oxpig/ANARCI.git /content/ANARCI_src
    !cd /content/ANARCI_src && python setup.py install -q
    !pip install -q abnumber
    from abnumber import Chain
    print("abnumber import OK (GitHub source)")

In [ ]:
# Cell 1b — sanity check on trastuzumab VH
from abnumber import Chain
tras = "EVQLVESGGGLVQPGGSLRLSCAASGFNIKDTYIHWVRQAPGKGLEWVARIYPTNGYTRYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCSRWGGDGFYAMDYWGQGTLVTVSS"
c = Chain(tras, scheme="imgt")
print("CDR1:", c.cdr1_seq)
print("CDR2:", c.cdr2_seq)
print("CDR3:", c.cdr3_seq)

In [ ]:
import urllib.request, os

CHEN_URL = "https://github.com/Tessier-Lab-UMich/Human_Ab_Polyreactivity/archive/refs/tags/v1.2.0-alpha.zip"
urllib.request.urlretrieve(CHEN_URL, "chen.zip")
!unzip -q -o chen.zip
CHEN_DIR = "Human_Ab_Polyreactivity-1.2.0-alpha"
assert os.path.isdir(CHEN_DIR), "Chen download/unzip failed"
print("Chen files:", os.listdir(CHEN_DIR)[:6])

RELEASE_TAG = "v1.4.2"
COEF_URL = (
    "https://raw.githubusercontent.com/"
    "avnish-deobhakta/polyatlas-peds/"
    f"{RELEASE_TAG}/models/model1_coefficients.csv"
)
urllib.request.urlretrieve(COEF_URL, "model1_coefficients.csv")
import hashlib
EXPECTED_COEF_SHA256 = "b12d35987312a76d1bd1773e9e2ede7637d1d21031983b41f97f3f74d7d71bd9"
_got = hashlib.sha256(open("model1_coefficients.csv","rb").read()).hexdigest()
assert _got == EXPECTED_COEF_SHA256, f"coefficient file checksum mismatch: {_got}"
print("model1_coefficients.csv checksum verified")
print("coefficients downloaded")

In [ ]:
import numpy as np, pandas as pd

KYTE_DOOLITTLE = {"A":1.8,"C":2.5,"D":-3.5,"E":-3.5,"F":2.8,"G":-0.4,"H":-3.2,"I":4.5,"K":-3.9,"L":3.8,"M":1.9,"N":-3.5,"P":-1.6,"Q":-3.5,"R":-4.5,"S":-0.8,"T":-0.7,"V":4.2,"W":-0.9,"Y":-1.3}
CHARGE_AT_PH74 = {"D":-1,"E":-1,"K":1,"R":1,"H":0.1}
AROMATIC=set("FWY"); POSITIVE=set("KR"); NEGATIVE=set("DE"); HYDROPHOBIC=set("ILVFMWYC")
PKA={"C_term":3.55,"D":4.05,"E":4.45,"H":5.98,"K":10.0,"R":12.0,"Y":10.0,"C":9.0,"N_term":8.0}

def net_charge(s):
    if not isinstance(s,str) or not s: return np.nan
    return sum(CHARGE_AT_PH74.get(a,0) for a in s.upper())

def frac(s,sub):
    if not isinstance(s,str) or not s: return np.nan
    s=s.upper()
    return sum(1 for a in s if a in sub)/len(s)

def frac_res(s,r):
    if not isinstance(s,str) or not s: return np.nan
    return s.upper().count(r)/len(s)

def mean_hphob(s):
    if not isinstance(s,str) or not s: return np.nan
    return np.mean([KYTE_DOOLITTLE.get(a,0) for a in s.upper()])

def max_hphob_run(s):
    if not isinstance(s,str) or not s: return 0
    best=cur=0
    for a in s.upper():
        if a in HYDROPHOBIC:
            cur+=1; best=max(best,cur)
        else:
            cur=0
    return best

def charge_dipole(s):
    if not isinstance(s,str) or len(s)<4: return 0
    m=len(s)//2
    return net_charge(s[:m])-net_charge(s[m:])

def estimate_pI(s):
    if not isinstance(s,str) or not s: return np.nan
    s=s.upper()
    def c_at(ph):
        c=1/(1+10**(ph-PKA["N_term"]))-1/(1+10**(PKA["C_term"]-ph))
        for a in s:
            if a in ("K","R"): c+=1/(1+10**(ph-PKA[a]))
            elif a in ("D","E"): c-=1/(1+10**(PKA[a]-ph))
            elif a=="H": c+=1/(1+10**(ph-PKA["H"]))
            elif a=="Y": c-=1/(1+10**(PKA["Y"]-ph))
            elif a=="C": c-=1/(1+10**(PKA["C"]-ph))
        return c
    lo,hi=0.0,14.0
    for _ in range(50):
        m=(lo+hi)/2
        if c_at(m)>0: lo=m
        else: hi=m
    return (lo+hi)/2

def build_features(df):
    f=pd.DataFrame(index=df.index)
    for region,col in [("H1","CDR1_nogaps"),("H2","CDR2_nogaps"),("H3","CDR3_nogaps"),("full","seq")]:
        s=df[col].fillna("").astype(str)
        f[f"{region}_len"]=s.str.len()
        f[f"{region}_charge"]=s.apply(net_charge)
        f[f"{region}_abs_charge"]=f[f"{region}_charge"].abs()
        f[f"{region}_pos_frac"]=s.apply(lambda x:frac(x,POSITIVE))
        f[f"{region}_neg_frac"]=s.apply(lambda x:frac(x,NEGATIVE))
        f[f"{region}_hphob"]=s.apply(mean_hphob)
        f[f"{region}_hphob_frac"]=s.apply(lambda x:frac(x,HYDROPHOBIC))
        f[f"{region}_arom"]=s.apply(lambda x:frac(x,AROMATIC))
        f[f"{region}_W"]=s.apply(lambda x:frac_res(x,"W"))
        f[f"{region}_R"]=s.apply(lambda x:frac_res(x,"R"))
        f[f"{region}_V"]=s.apply(lambda x:frac_res(x,"V"))
        f[f"{region}_G"]=s.apply(lambda x:frac_res(x,"G"))
    f["H3_charge_dipole"]=df["CDR3_nogaps"].fillna("").astype(str).apply(charge_dipole)
    f["H3_max_hphob_run"]=df["CDR3_nogaps"].fillna("").astype(str).apply(max_hphob_run)
    f["H3_pI"]=df["CDR3_nogaps"].fillna("").astype(str).apply(estimate_pI)
    f["full_pI"]=df["seq"].fillna("").astype(str).apply(estimate_pI)
    return f.fillna(0)

print("feature code ready")

In [ ]:
from abnumber import Chain
import time

f = f"{CHEN_DIR}/Supplemental Datasets/Human Ab Poly Dataset S1_v2.xlsx"
df = pd.read_excel(f, sheet_name="Sheet1", header=2)
df["label"] = df["Name"].astype(str).str.contains("high", case=False).astype(int)

N_PER_CLASS = 40000  # 40,000 per class (balanced 80,000 sample); random_state=42 throughout
pos = df[df.label==1].sample(n=min(N_PER_CLASS,(df.label==1).sum()), random_state=42)
neg = df[df.label==0].sample(n=min(N_PER_CLASS,(df.label==0).sum()), random_state=42)
sub = pd.concat([pos,neg]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Subsampled to {len(sub)}")

def anarci_cdrs(vh):
    try:
        c = Chain(str(vh), scheme="imgt")
        return c.cdr1_seq, c.cdr2_seq, c.cdr3_seq
    except Exception:
        return None, None, None

t=time.time(); h1=[]; h2=[]; h3=[]
for i,vh in enumerate(sub["VH"].astype(str)):
    a,b,cc = anarci_cdrs(vh)
    h1.append(a); h2.append(b); h3.append(cc)
    if i%5000==0 and i>0:
        print(f"  {i}/{len(sub)}  ({time.time()-t:.0f}s)")
sub["CDR1_nogaps"]=h1; sub["CDR2_nogaps"]=h2; sub["CDR3_nogaps"]=h3
sub["seq"]=sub["VH"].astype(str)
n0=len(sub)
sub=sub[sub.CDR1_nogaps.notna()&sub.CDR2_nogaps.notna()&sub.CDR3_nogaps.notna()].reset_index(drop=True)
print(f"ANARCI annotated {len(sub)}/{n0} ({100*len(sub)/n0:.1f}%) in {time.time()-t:.0f}s")
print("label balance:", sub.label.value_counts().to_dict())
sub.to_csv("chen_annotated.csv", index=False)
print("saved chen_annotated.csv")

In [ ]:
import pandas as pd, numpy as np, json
try:
    sub
except NameError:
    sub = pd.read_csv('chen_annotated.csv')
sub = sub[sub.CDR1_nogaps.notna() & sub.CDR2_nogaps.notna() & sub.CDR3_nogaps.notna()].reset_index(drop=True)

# Cell 3 — ONE stratified 80/20 split; evaluate EVERY condition on the common test set
import urllib.request
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

X = build_features(sub); y = sub["label"].values

# frozen Model 1 coefficients — already downloaded and checksum-verified in Cell 3
coef = pd.read_csv("model1_coefficients.csv")
intercept = float(coef[coef.feature=="_intercept"].standardized_coefficient.iloc[0])
coef = coef[coef.feature!="_intercept"].copy(); m1 = list(coef.feature)

# THE common split — all bars evaluated on idx_test
idx = np.arange(len(X))
idx_tr, idx_te = train_test_split(idx, test_size=0.2, random_state=42, stratify=y)
Xte_all = X.iloc[idx_te]; yte = y[idx_te]
N_TEST = len(idx_te)
print(f"common held-out test set: n={N_TEST}, pos rate={yte.mean():.3f}")

def frozen_score(Xdf):
    s = np.full(len(Xdf), intercept, float)
    for _,cr in coef.iterrows():
        s += cr.standardized_coefficient*((Xdf[cr.feature].values-cr.train_mean)/cr.train_std)
    return s

def boot_auroc(yv, sv, n=500, seed=42):
    rng=np.random.RandomState(seed); out=[]
    for _ in range(n):
        b=rng.choice(len(yv),len(yv),replace=True)
        try: out.append(roc_auc_score(yv[b], sv[b]))
        except: pass
    return float(np.percentile(out,2.5)), float(np.percentile(out,97.5))

results={"n_test": int(N_TEST), "test_pos_rate": float(yte.mean())}

# (A) Frozen Model 1 — evaluated ON THE COMMON TEST SET (not all 80k)
sA = frozen_score(Xte_all)
results["frozen"]={"auroc":roc_auc_score(yte,sA),"auprc":average_precision_score(yte,sA),
                   "auroc_ci":boot_auroc(yte,sA)}

# (B) Refit 13-feature family — fit on train, eval on common test
scB=StandardScaler().fit(X.iloc[idx_tr][m1].values)
lrB=LogisticRegression(class_weight="balanced",random_state=42,max_iter=500).fit(scB.transform(X.iloc[idx_tr][m1].values), y[idx_tr])
sB=lrB.predict_proba(scB.transform(Xte_all[m1].values))[:,1]
results["refit13"]={"auroc":roc_auc_score(yte,sB),"auprc":average_precision_score(yte,sB),"auroc_ci":boot_auroc(yte,sB)}

# (C) Refit full 52 — fit on train, eval on common test
scC=StandardScaler().fit(X.iloc[idx_tr].values)
lrC=LogisticRegression(class_weight="balanced",random_state=42,max_iter=500).fit(scC.transform(X.iloc[idx_tr].values), y[idx_tr])
sC=lrC.predict_proba(scC.transform(Xte_all.values))[:,1]
results["refit52"]={"auroc":roc_auc_score(yte,sC),"auprc":average_precision_score(yte,sC),"auroc_ci":boot_auroc(yte,sC)}

# charge baselines — ON THE COMMON TEST SET
for name,col in [("full_charge","full_charge"),("full_pi","full_pI"),("h3_charge","H3_charge")]:
    sc_=Xte_all[col].values
    results[name]={"auroc":roc_auc_score(yte,sc_),"auprc":average_precision_score(yte,sc_)}

# precision at top-5% / top-10% (report precision directly), on common test set, for frozen & refit13
def topk_precision(yv, sv, k):
    order=np.argsort(-sv); kk=int(k*len(yv)); return float(yv[order[:kk]].mean())
results["frozen"]["prec_top5"]=topk_precision(yte,sA,0.05)
results["frozen"]["prec_top10"]=topk_precision(yte,sA,0.10)
results["refit13"]["prec_top5"]=topk_precision(yte,sB,0.05)
results["refit13"]["prec_top10"]=topk_precision(yte,sB,0.10)

# coefficient signs (refit13 vs NbBench) unchanged concept, recompute on this fit
results["refit_coef"]={k:float(v) for k,v in zip(m1, lrB.coef_[0])}
results["nbbench_coef"]={f:float(coef[coef.feature==f].standardized_coefficient.iloc[0]) for f in m1}

print(json.dumps({k:(v if not isinstance(v,dict) else {kk:round(vv,4) if isinstance(vv,float) else vv for kk,vv in v.items()}) for k,v in results.items() if k in ["n_test","test_pos_rate","frozen","refit13","refit52","full_charge","full_pi","h3_charge"]}, indent=2))
json.dump(results, open("chen_fig5_common_test.json","w"), indent=2)
print("\nsaved chen_fig5_common_test.json")

In [ ]:
# Cell 4 — full-data frozen/baseline scores + write the two JSON files the workflow needs
# (1) score frozen Model 1 and charge baselines on ALL annotated antibodies (n=79,999),
# (2) save chen_anarci_results.json (all-data external-validation results),
# (3) combine full-data + common-test + coefficients into chen_fig5_merged.json
#     (this merged file is what make_figure5.py reads to build Figure 5).

# --- full-data (all annotated) frozen + baselines ---
sA_full = frozen_score(X)                      # X = features for all annotated antibodies
frozen_full_auroc = roc_auc_score(y, sA_full)
frozen_full_auprc = average_precision_score(y, sA_full)
frozen_full_ci    = boot_auroc(y, sA_full)

full_baselines = {}
full_baselines_auprc = {}
for name, col in [("full_charge","full_charge"), ("full_pi","full_pI"), ("h3_charge","H3_charge")]:
    full_baselines[name]       = float(roc_auc_score(y, X[col].values))
    full_baselines_auprc[name] = float(average_precision_score(y, X[col].values))

# Coefficient panel (Figure 5C) uses the coefficients fit on the Chen 80% TRAINING
# partition (already computed in the common-test evaluation above and stored in
# results["refit_coef"]). We do NOT refit on all 79,999 here, to keep the figure's
# coefficient source consistent with what make_figure5.py reads.
nbbench_coef = {f: float(coef[coef.feature==f].standardized_coefficient.iloc[0]) for f in m1}

# --- (2) chen_anarci_results.json : all-data external-validation results (n=79,999) ---
# chen_anarci_results.json is limited to FULL-DATA frozen and baseline results
# (all n_full annotated antibodies). Common-test refit results live in
# chen_fig5_common_test.json / chen_fig5_merged.json, so no single denominator
# is ambiguous here.
chen_anarci_results = {
    "n_full": int(len(X)),
    "note": "All metrics in this file are computed on the full annotated set (n_full). Common-test refit results are in chen_fig5_common_test.json and chen_fig5_merged.json.",
    "frozen_full": {"auroc": float(frozen_full_auroc), "auprc": float(frozen_full_auprc), "auroc_ci": list(frozen_full_ci)},
    "charge_baselines_auroc": full_baselines,
    "charge_baselines_auprc": full_baselines_auprc,
    "nbbench_coef": nbbench_coef,
}
json.dump(chen_anarci_results, open("chen_anarci_results.json", "w"), indent=2)
print("saved chen_anarci_results.json (all-data, n=%d): frozen AUROC %.4f" % (len(X), frozen_full_auroc))

# --- (3) chen_fig5_merged.json : what make_figure5.py reads ---
merged = {
    "n_full": int(len(X)),
    "n_test": int(results["n_test"]),
    "test_pos_rate": float(results["test_pos_rate"]),
    "full_data": {
        "frozen": {"auroc": float(frozen_full_auroc), "auroc_ci": list(frozen_full_ci), "auprc": float(frozen_full_auprc)},
        "full_charge": full_baselines["full_charge"], "full_pi": full_baselines["full_pi"], "h3_charge": full_baselines["h3_charge"],
        "full_charge_auprc": full_baselines_auprc["full_charge"], "full_pi_auprc": full_baselines_auprc["full_pi"], "h3_charge_auprc": full_baselines_auprc["h3_charge"],
    },
    "common_test": results,
}
json.dump(merged, open("chen_fig5_merged.json", "w"), indent=2)
print("saved chen_fig5_merged.json (make_figure5.py input)")
print("\nAll three JSON files written: chen_anarci_results.json, chen_fig5_common_test.json, chen_fig5_merged.json")

Run-all writes three files, all needed downstream:

- **`chen_anarci_results.json`** — all-data (n=79,999) frozen and baseline results.
- **`chen_fig5_common_test.json`** — common held-out test set (n=16,000) results.
- **`chen_fig5_merged.json`** — combined file read by `make_figure5.py` to build Figure 5.

Download all three (folder icon, left) and send them back to regenerate Figure 5 and the manuscript numbers.